# Ранжирование фрагментов по релевантности

AMML Ranking Competition 2026. На входе CSV конкурса или ZIP с ними, на выходе `submission.csv` со столбцами `id,prediction` в порядке образца. Ячейки исполняются сверху вниз в ядре Python на Linux с двумя GPU NVIDIA T4. Пакеты перечислены в `requirements.txt`.

В Kaggle подключите данные соревнования как Input. Публичные веса можно заранее подключить как Input или загрузить с Hugging Face при включённом Internet. Локально поместите данные в `data/` рядом с ноутбуком или задайте путь в `AMML_INPUT_DIR`.


In [ ]:
import os
import sys
import time
import shutil
import signal
import tempfile
import subprocess
from pathlib import Path

BASE = Path.cwd().resolve()
START = time.monotonic()
LIMIT = 8 * 3600
kaggle = Path("/kaggle/input").is_dir()
INPUT = Path(os.environ.get("AMML_INPUT_DIR", "/kaggle/input" if kaggle else BASE / "data")).resolve()
OUTPUT = Path(os.environ.get("AMML_OUTPUT_DIR", "/kaggle/working" if kaggle else BASE)).resolve()
if not INPUT.exists():
    raise FileNotFoundError(f"Не найдены конкурсные данные в {INPUT}.")
OUTPUT.mkdir(parents=True, exist_ok=True)
temp_root = Path("/kaggle/temp") if kaggle else Path(tempfile.gettempdir())
temp_root.mkdir(parents=True, exist_ok=True)
temp = tempfile.TemporaryDirectory(prefix="amml-final-", dir=temp_root)
WORK = Path(temp.name)
SRC = WORK / "src"
SRC.mkdir()
# Each %%writefile cell below creates a temporary importable module for both GPU processes.
os.chdir(SRC)

# True — выполнить 306 шагов обучения. False — сразу применить конкурсный адаптер.
RETRAIN = True


## Данные и лексическая модель

Очистка фрагментов и заголовков, разделение связанных запросов и построение признаков совпадений. Градиентный бустинг учится на размеченной обучающей части и оценивает кандидатов для каждого запроса.


In [ ]:
%%writefile data.py
"""Contest rows, grouped validation, hard-negative scores, and atomic submission output."""
from collections import Counter
from contextlib import contextmanager
from functools import lru_cache
from pathlib import Path
import hashlib
import html
import math
import os
import re
import tempfile
import unicodedata
import zipfile

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from threadpoolctl import threadpool_limits


@contextmanager
def _csv_stream(source):
    outer, member = source
    if member is None:
        with outer.open("rb") as stream:
            yield stream
    else:
        with zipfile.ZipFile(outer) as archive, archive.open(member) as stream:
            yield stream


def _read_competition(input_root):
    root = Path(input_root).expanduser()
    if not root.exists():
        raise FileNotFoundError(f"Путь к конкурсным данным не найден: {root}")
    kinds = ("train", "test", "sample_submission")
    patterns = {kind: re.compile(re.escape(kind) + r"(?:\s*\(\d+\))*\.csv", re.I) for kind in kinds}
    candidates = {kind: [] for kind in kinds}
    archives = []

    def add(path, member=None):
        name = Path(member or path).name
        for kind, pattern in patterns.items():
            if pattern.fullmatch(name):
                candidates[kind].append((path, member))

    if root.is_file():
        archives.append(root)
    else:
        visited = set()
        for folder, dirs, names in os.walk(root, followlinks=True):
            real = os.path.realpath(folder)
            if real in visited:
                dirs.clear()
                continue
            visited.add(real)
            for name in names:
                path = Path(folder) / name
                if path.suffix.lower() == ".zip":
                    archives.append(path)
                elif path.suffix.lower() == ".csv":
                    add(path)

    if not all(candidates.values()):
        for path in sorted(archives):
            if not zipfile.is_zipfile(path):
                continue
            with zipfile.ZipFile(path) as archive:
                for name in archive.namelist():
                    if "__MACOSX/" not in name and name.lower().endswith(".csv"):
                        add(path, name)
    base = {"id", "query_id", "document_id", "article_id", "query", "title", "text",
            "passage_index", "article_passage_count", "article_char_count", "ingest_source"}
    required = {"train": base | {"relevance"}, "test": base, "sample_submission": {"id", "prediction"}}
    frames = []
    for kind in kinds:
        matching = []
        for source in candidates[kind]:
            with _csv_stream(source) as stream:
                columns = set(pd.read_csv(stream, nrows=0).columns)
            if required[kind] <= columns and (kind != "test" or "relevance" not in columns):
                matching.append(source)
        if not matching:
            raise FileNotFoundError(f"Подключите конкурсные CSV (или их ZIP): не найден {kind}.csv в {root}")
        if len(matching) > 1:
            digests = set()
            for source in matching:
                digest = hashlib.sha256()
                with _csv_stream(source) as stream:
                    for data in iter(lambda: stream.read(1024 * 1024), b""):
                        digest.update(data)
                digests.add(digest.digest())
            if len(digests) > 1:
                raise ValueError(f"Найдено несколько разных {kind}.csv. Укажите папку или ZIP этого конкурса.")
        with _csv_stream(matching[0]) as stream:
            frames.append(pd.read_csv(stream, dtype=str, keep_default_na=False))
    train, test, sample = frames
    for frame in frames:
        if frame.empty or frame.id.eq("").any() or not frame.id.is_unique:
            raise ValueError("Пустая таблица, пропущенные или дублирующиеся id.")
    if set(train.id) & set(test.id) or set(test.id) != set(sample.id):
        raise ValueError("Проверьте файлы: train, test и sample_submission относятся к разным выборкам.")
    if set(train.relevance) != {"0","1"} or any(f.query_id.eq("").any() for f in (train, test)):
        raise ValueError("Нужны бинарные relevance и заполненные query_id.")
    return train, test, sample


def amml_norm(value):
    value=unicodedata.normalize("NFKC",value).casefold().replace("ё","е").replace("\u0301","")
    return " ".join(re.findall(r"[^\W_]+",value,flags=re.UNICODE))

def amml_visible(value):
    value=html.unescape(html.unescape(value))
    value=re.sub(r"</?(?:p|br|div|span)(?:\s[^>]{0,150})?\s*/?>"," ",value,flags=re.I)
    value=re.sub(r"^(?:Главная › Энциклопедия|Навигация: Главная \| Статьи)\s+","",value)
    value=re.sub(r"\s+(?:Все материалы сайта|Политика конфиденциальности \| Обратная связь)$","",value)
    return " ".join(unicodedata.normalize("NFC",value).translate({0x00ad:None,0x200b:None,0xfeff:None,0x301:None}).split())

def amml_prepare(data):
    d=data.copy().reset_index(drop=True)
    for field in ("query","title","text"):
        lu={s:amml_visible(s) for s in d[field].unique()}
        d[field+"_clean"]=d[field].map(lu)
    titles=d.title_clean.str.replace(r"\s+[-—] Википедия$","",regex=True)
    titles=titles.mask(titles.str.casefold().isin({"","null","n/a","не указано","—"}))
    # Доступно при обработке каждого полного списка кандидатов нового запроса.
    by=titles.groupby([d.query_id,d.article_id],sort=False)
    one=by.transform("nunique").eq(1)
    d["title_clean"]=titles.fillna(by.transform("first").where(one)).fillna("")
    return d

_TREE_STOP = frozenset("""а без будет были был была было быть в во вот где для до его ее если есть же за из и или из-за к как когда какой какая какие какое каком какого какую каким кем который которая которое которые которого куда ли на над нас не него нее нет ни но о об от почему по под при про с со сколько та так также там те тем того то той том тот ту у уже что чем чей чья чтобы эта эти это этот этих является""".split())
_TREE_END = re.compile(r"(иями|ями|ами|его|ого|ему|ому|ыми|ими|ую|юю|ая|яя|ые|ие|ое|ее|ой|ый|ий|ым|им|ом|ем|ах|ях|ам|ям|ов|ев|ей|ла|ли|ло|ет|ут|ют|ит|ат|ят|ть|ых|их|а|я|о|е|и|ы|у|ю|ь)$")
_TREE_TOK = re.compile(r"[а-яa-z0-9]+", re.I)
_TREE_YEAR = re.compile(r"(?<!\d)(?:1\d{3}|20\d{2})(?!\d)")
_TREE_PRONOUN = re.compile(r"^(он|она|оно|они|этот|эта|это|после|затем|однако|также|кроме|поэтому|там)\b", re.I)

@lru_cache(160000)
def tree_stem(word):
    if len(word) < 4 or not word.isalpha():
        return word
    stem = _TREE_END.sub("", word, count=1)
    return stem if len(stem) >= 3 else word

def tree_tokens(value):
    return _TREE_TOK.findall(str(value).casefold().replace("ё","е").replace("\u0301",""))

def tree_num(value, default=-1.):
    s = re.sub(r"\s", "", str(value)).replace(",",".")
    try:
        n=float(s)
        return n if np.isfinite(n) else default
    except ValueError:
        return default

def tree_text_features(q, title, text, stem_text, stem_title, bm25_df, group_n, avlen):
    q_set = set(q)
    qs = [tree_stem(w) for w in q if w not in _TREE_STOP]
    if not qs:
        qs = [tree_stem(w) for w in q]
    terms = sorted(set(qs))
    weights = {t: math.log1p((group_n-bm25_df.get(t,0)+.5)/(bm25_df.get(t,0)+.5)) for t in terms}
    wt = sum(weights.values()) or 1.
    counts = Counter(stem_text)
    features = {}
    for name, words in (("t",title),("b",text)):
        ws = set(words)
        features[name+"_exact"] = len(ws & q_set)/max(1,len(q_set))
        features[name+"_exact_content"] = sum(w in ws for w in q if w not in _TREE_STOP)/max(1,sum(w not in _TREE_STOP for w in q))
    for name, words in (("t",stem_title),("b",stem_text),("lead",stem_text[:28])):
        ws = set(words)
        hits=[t for t in terms if t in ws]
        features[name+"_hits"]=len(hits)
        features[name+"_cov"]=len(hits)/max(1,len(terms))
        features[name+"_weighted"]=sum(weights[t] for t in hits)/wt
        features[name+"_rarest"]=max((weights[t] for t in hits),default=0.)
        features[name+"_jacc"]=len(hits)/max(1,len(ws.union(terms)))
    features["both_cov"]=sum(t in counts or t in stem_title for t in terms)/max(1,len(terms))
    features["b_tf"]=sum(math.log1p(counts[t])*weights[t] for t in terms)/wt
    title_tf=Counter(stem_title)
    bm25=0.
    for t in terms:
        tf = counts[t] + 2.0*title_tf[t]
        if tf:
            bm25 += weights[t]*tf/(tf+1.2*(.25+.75*len(text)/max(1,avlen)))*2.2
    features["bm25"]=bm25
    # Близость терминов запроса в тексте и лучшем окне шириной 16/48 слов.
    hits=[(i,t) for i,t in enumerate(stem_text) if t in weights]
    features["first_match"]=hits[0][0]/max(1,len(text)) if hits else 1.1
    features["match_span"]=(hits[-1][0]-hits[0][0])/max(1,len(text)) if hits else 1.1
    for width in [16,48]:
        active=Counter()
        left=0
        best_weight=best_count=total=0.
        for right, (pos,t) in enumerate(hits):
            if not active[t]: total+=weights[t]
            active[t]+=1
            while pos-hits[left][0]>=width:
                other=hits[left][1]
                active[other]-=1
                if not active[other]:
                    total-=weights[other]
                    del active[other]
                left+=1
            best_weight=max(best_weight,total)
            best_count=max(best_count,len(active))
        features[f"win{width}_w"]=best_weight/wt
        features[f"win{width}_cov"]=best_count/max(1,len(terms))
    query_pairs=set(zip(qs,qs[1:]))
    for name, words in (("t",stem_title),("b",stem_text)):
        filtered=[w for w in words if w in weights]
        contiguous=sum(1 for pair in zip(words,words[1:]) if pair in query_pairs)
        ordered=sum(1 for pair in zip(filtered,filtered[1:]) if pair in query_pairs)
        features[name+"_bigram"]=min(1.,contiguous/max(1,len(query_pairs)))
        features[name+"_ordered"]=math.log1p(ordered)
    for size in (3,5):
        prefixes={word[:size] for word in q if word not in _TREE_STOP and len(word)>=size}
        for name,words in (("t",title),("b",text)):
            available={w[:size] for w in words}
            features[name+"_prefix"+str(size)] = len(available & prefixes)/max(1,len(prefixes))
    return features

def tree_matrix(frame):
    raw=frame.reset_index(drop=True)
    output=[]
    for _, group in raw.groupby("query_id",sort=False):
        docs=list(group.itertuples())
        q=tree_tokens(docs[0].query_clean)
        txts=[tree_tokens(row.text_clean) for row in docs]
        titles=[tree_tokens(row.title_clean) for row in docs]
        stem_txts=[[tree_stem(t) for t in words] for words in txts]
        stem_titles=[[tree_stem(t) for t in words] for words in titles]
        dfs=Counter(t for body,title in zip(stem_txts,stem_titles) for t in set(body+title))
        mean_len=np.mean([len(v) for v in txts])
        count_article=Counter(r.article_id for r in docs)
        total=len(docs)
        group_pos={r.Index:tree_num(r.passage_index)-int(r.ingest_source=="S3") for r in docs}
        for j,r in enumerate(docs):
            text=str(r.text_clean)
            rawtext=str(r.text)
            rawtitle=str(r.title)
            query=str(r.query_clean).casefold()
            body=txts[j]
            features=tree_text_features(q,titles[j],body,stem_txts[j],stem_titles[j],dfs,total,mean_len)
            position=group_pos[r.Index]
            n_pass=tree_num(r.article_passage_count)
            chars=tree_num(r.article_char_count)
            n_tokens=len(body)
            n_numbers=sum(w.isdigit() for w in body)
            n_years=len(_TREE_YEAR.findall(text))
            neighbours=[abs(position-group_pos[v.Index]) for v in docs if v.Index!=r.Index and v.article_id==r.article_id and group_pos[v.Index]>=0 and position>=0]
            features.update({
                "n":total, "q_len":len(q),"q_chars":len(query),
                "title_len":len(titles[j]),"body_len":n_tokens,"chars":len(text),
                "avg_word":sum(map(len,body))/max(1,n_tokens),
                "num_tokens":n_numbers,"years":n_years,"num_density":n_numbers/max(1,n_tokens),
                "unique_frac":len(set(body))/max(1,n_tokens),"caps":sum(c.isupper() for c in rawtext),
                "quoted":int('«' in text or '"' in text),"sentences":text.count("."),
                "pronoun_opening":int(bool(_TREE_PRONOUN.search(text.lstrip()))),
                "source":ord(str(r.ingest_source)[-1]) if r.ingest_source else 0,
                "title_absent":int(rawtitle.strip().casefold() in {"","null","n/a","не указано","—"}),
                "html_noise":int("<p>" in rawtext or "&lt;" in rawtext or "&amp;" in rawtext),
                "invisible":int(any(v in rawtext for v in ("\u200b","\ufeff","\u00ad"))),
                "p_idx":position,"p_count":n_pass,"p_ratio":position/n_pass if n_pass>0 and position>=0 else -1,
                "p_first":int(position==0),"p_last":int(position>=0 and n_pass>0 and position==n_pass-1),
                "art_chars":np.log1p(max(chars,0)),"art_scale_flag":int(chars>1e6 and chars%1000==0),
                "art_chars_per_p":chars/n_pass if n_pass>0 and chars>0 else -1,
                "n_article":len(count_article),"article_freq":count_article[r.article_id],
                "near_fragment":min(neighbours,default=-1),
            })
            for key,pat in (("date",r"когда|каком году|какого числа|каком веке"),
                            ("qty",r"сколько|каков[ао]? (?:рост|высота|длина)|какую высоту"),
                            ("who",r"^кто\b|^как звали"),("where",r"^где\b|^в каком городе"),
                            ("bool",r"\bли\b"),("why",r"^почему")):
                features["q_"+key]=int(bool(re.search(pat,query)))
            output.append((r.Index,features))
    features=pd.DataFrame([v for _,v in output],index=[i for i,_ in output]).sort_index()
    query=raw["query_id"]
    article=raw["article_id"]
    signals=["bm25","t_weighted","b_weighted","b_exact","t_exact","t_bigram","b_bigram","win16_w","b_tf",
             "both_cov","p_idx","body_len","title_len","article_freq","years","p_ratio"]
    by_q=features[signals].groupby(query,sort=False)
    ranks=by_q.rank(method="average",pct=True).add_suffix("_q_rank")
    top_gap=(features[signals]-by_q.transform("max")).add_suffix("_q_top_gap")
    art_signals=["bm25","t_weighted","b_weighted","win16_w","b_tf","both_cov","p_idx"]
    by_a=features[art_signals].groupby([query,article],sort=False)
    art_max=by_a.transform("max")
    art_mean=by_a.transform("mean")
    return pd.concat([features,ranks,top_gap,(features[art_signals]-art_max).add_suffix("_art_gap"),
                      art_mean.add_suffix("_art_mean")],axis=1).fillna(-1.).astype("float32")

def _folds(frame):
    """Keep connected queries, passages and shared documents on the same side."""
    query_codes, queries = pd.factorize(frame.query_id)
    offset = len(queries)
    left, right = [], []
    for values in (frame.document_id, frame.query_clean, frame.text_clean):
        if values.name != "document_id":
            lookup = {s: amml_norm(s) for s in values.unique()}
            values = values.map(lookup)
        codes, keys = pd.factorize(values.mask(values.eq("")))
        known = codes >= 0
        left.append(query_codes[known])
        right.append(offset + codes[known])
        offset += len(keys)
    a, b = np.concatenate(left), np.concatenate(right)
    graph = coo_matrix((np.ones(len(a), dtype=np.uint8), (a,b)), shape=(offset,offset)).tocsr()
    components = connected_components(graph, directed=False)[1][query_codes]
    count = min(5, len(np.unique(components)))
    if count < 2:
        raise ValueError("Слишком мало независимых групп запросов для валидации.")
    folds = np.full(len(frame), -1, dtype=np.int8)
    for number, (_, valid) in enumerate(GroupKFold(count).split(frame, groups=components)):
        folds[valid] = number
    return folds


def within_query(frame, scores):
    values = pd.Series(np.asarray(scores, dtype=np.float64))
    if len(values) != len(frame) or not np.isfinite(values).all():
        raise ValueError("Число конечных оценок должно совпадать с числом кандидатов.")
    group = values.groupby(frame.query_id.to_numpy(), sort=False)
    return ((values - group.transform("mean")) / group.transform("std", ddof=0).replace(0.,1.)).to_numpy()


def ndcg(frame, scores):
    """Mean per-query binary NDCG@10, deterministic string-ID tie breaks, zero for IDCG=0."""
    values = np.asarray(scores, dtype=float)
    if values.shape != (len(frame),) or not np.isfinite(values).all() or not len(frame):
        raise ValueError("Недопустимые оценки или пустая таблица для NDCG.")
    ranked = frame[["query_id","id","label"]].assign(score=values).sort_values(
        ["query_id","score","id"], ascending=[True,False,True])
    groups = ranked.groupby("query_id", sort=False)
    rank = groups.cumcount().to_numpy()
    gain = ranked.label.to_numpy() * (rank < 10) / np.log2(rank + 2)
    dcg = pd.Series(gain).groupby(ranked.query_id.to_numpy(), sort=False).sum().to_numpy()
    ideal = np.r_[0., np.cumsum(1. / np.log2(np.arange(2,12)))]
    denominator = ideal[np.minimum(groups.label.sum().to_numpy(dtype=int), 10)]
    return float(np.divide(dcg, denominator, out=np.zeros_like(dcg), where=denominator > 0).mean())


def save_submission(test_frame, sample_frame, scores, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pred = np.asarray(scores, dtype=float)
    order = pd.Index(test_frame.id)
    if not order.is_unique or not sample_frame.id.is_unique or len(test_frame) != len(sample_frame):
        raise ValueError("Требуются уникальные id и одинаковое число строк в test и образце.")
    ix = order.get_indexer(sample_frame.id)
    if pred.shape != (len(test_frame),) or (ix < 0).any() or not np.isfinite(pred).all():
        raise ValueError("Число оценок, id или значения prediction некорректны.")
    output = pd.DataFrame({"id": sample_frame.id.to_numpy(), "prediction":pred[ix]})
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".tmp", delete=False) as stream:
        temp = Path(stream.name)
    try:
        output.to_csv(temp, index=False)
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def prepare_data(input_root, work_dir):
    """Fit the supervised lexical scorer and prepare per-query encoder inputs."""
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    train, test, sample = _read_competition(input_root)
    record_input_digest(train, test, sample, work_dir)
    train, test = amml_prepare(train), amml_prepare(test)
    train["label"] = train.relevance.astype(np.int8)
    train["fold"] = _folds(train)
    print(f"Конкурс: {len(train):,} train, {len(test):,} test. Считаю признаки CPU…", flush=True)
    matrices = [tree_matrix(frame) for frame in (train, test)]
    # Train hard-negative mining and all validation scores exclude holdout labels.
    fit = train.fold.ne(0).to_numpy()
    model = HistGradientBoostingClassifier(max_iter=200, max_leaf_nodes=23,
        learning_rate=.065, min_samples_leaf=35, l2_regularization=3., early_stopping=False,
        random_state=1949)
    with threadpool_limits(limits=min(4, os.cpu_count() or 1)):
        model.fit(matrices[0].loc[fit], train.loc[fit,"label"])
        for frame, matrix in zip((train,test), matrices):
            frame["lex"] = model.decision_function(matrix)
            frame["query"] = frame.query_clean
            frame["passage"] = (frame.title_clean.where(frame.title_clean.eq(""),frame.title_clean+"\n")
                                + frame.text_clean)
    fields = ["id","query_id","query","passage","lex","document_id","article_id"]
    train[fields + ["label","fold"]].to_pickle(work_dir / "train.pkl")
    test[fields].to_pickle(work_dir / "test.pkl")
    sample[["id","prediction"]].to_pickle(work_dir / "sample.pkl")

def record_input_digest(train, test, sample, work):
    digest = hashlib.sha256()
    for frame in (train, test, sample[["id"]]):
        columns = sorted(frame.columns)
        digest.update(("\x1e".join(columns) + "\n").encode("utf-8"))
        rows = pd.util.hash_pandas_object(frame[columns], index=False).to_numpy(dtype="<u8")
        digest.update(rows.tobytes())
    (work/"inputs.sha256").write_text(digest.hexdigest())


def prepare_inference_inputs(input_root, work):
    train, test, sample = _read_competition(input_root)
    record_input_digest(train, test, sample, work)
    train, test = amml_prepare(train), amml_prepare(test)
    train["label"] = train.relevance.astype(np.int8)
    train["fold"] = _folds(train)
    for frame in (train, test):
        frame["query"] = frame.query_clean
        frame["passage"] = frame.title_clean.where(frame.title_clean.eq(""), frame.title_clean+"\n") + frame.text_clean
    columns = ["id","query_id","query","passage"]
    train[columns + ["label","fold"]].to_pickle(work/"train.pkl")
    test[columns].to_pickle(work/"test.pkl")
    sample[["id","prediction"]].to_pickle(work/"sample.pkl")


## Энкодер Giga

Запрос получает инструкцию поиска. Для каждого запроса и фрагмента усредняются выходы двунаправленного трансформера по непустым токенам. Косинусное сходство вычисляется по нормированным векторам. Обучаются адаптеры LoRA в проекциях Q и V.


In [ ]:
%%writefile encoder.py
"""Giga 3B (August 2026), with the publisher's bidirectional backbone."""

import json
import math
import os
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F


MODEL_ID = "ai-sage/Giga-Embeddings-instruct-3B-0826"
MODEL_REVISION = "b71168088212f0a13688514e4fc288a86106c4e4"
QUERY_INSTRUCTION = "Найди фрагмент текста, который отвечает на поисковый запрос."


def query_text(text):
    return f"Instruct: {QUERY_INSTRUCTION}\nQuery: {text}"


def _complete_snapshot(folder):
    """Accept official layout or shards of the same architecture."""
    try:
        config = json.loads((folder / "config.json").read_text())
        if (config.get("model_type"), config.get("hidden_size"), config.get("num_hidden_layers")) != (
            "qwen3_bidirec", 2048, 36
        ):
            return False
        required = [
            "tokenizer.json", "tokenizer_config.json",
            "configuration_qwen3_bidirectional.py", "modeling_qwen3_bidirectional.py"
        ]
        weights_index = folder / "model.safetensors.index.json"
        if weights_index.is_file():
            shards = set(json.loads(weights_index.read_text())["weight_map"].values())
            if not shards:
                return False
            required.extend(shards)
        else:
            required.append("model.safetensors")
        return all((folder / name).is_file() and (folder / name).stat().st_size > 0 for name in required)
    except (OSError, ValueError, KeyError, TypeError):
        return False


def prepare_model_dir(input_root, cache_dir, allow_download=True):
    """Resolve a mounted snapshot first. Only download the pinned public release."""
    from huggingface_hub import snapshot_download

    explicit = os.getenv("GIGA_MODEL_DIR")
    if explicit:
        folder = Path(explicit)
        if not _complete_snapshot(folder):
            raise ValueError(f"GIGA_MODEL_DIR не содержит полный срез {MODEL_ID}: {folder}")
        return str(folder)
    # Kaggle can expose mounted inputs through symbolic links. Directory IDs
    # prevent re-scanning the same mount or following a symlink cycle.
    seen = set()
    for directory, children, files in os.walk(input_root, topdown=True, followlinks=True):
        folder = Path(directory)
        try:
            stat = folder.stat()
        except OSError:
            children.clear()
            continue
        identity = (stat.st_dev, stat.st_ino)
        if identity in seen:
            children.clear()
            continue
        seen.add(identity)
        children.sort()
        if "config.json" in files and _complete_snapshot(folder):
            return str(folder)
    options = dict(repo_id=MODEL_ID, revision=MODEL_REVISION, cache_dir=str(cache_dir),
                   allow_patterns=["config.json", "configuration_qwen3_bidirectional.py",
                                   "modeling_qwen3_bidirectional.py", "tokenizer*.json",
                                   "model*.safetensors", "model.safetensors.index.json"])
    try:
        cached = Path(snapshot_download(**options, local_files_only=True))
        if _complete_snapshot(cached):
            return str(cached)
    except FileNotFoundError:
        pass
    if not allow_download:
        raise FileNotFoundError(
            f"Подключите в Kaggle Input каталог {MODEL_ID} со всеми весами, "
            "токенизатором и двумя файлами Python из официального репозитория."
        )
    folder = Path(snapshot_download(**options))
    if not _complete_snapshot(folder):
        raise RuntimeError(f"Неполный срез {MODEL_ID} в {folder}")
    return str(folder)


class LoRALinear(nn.Module):
    """Frozen projection plus low-rank update, with FP32 master parameters."""

    def __init__(self, base, rank=8, alpha=16):
        super().__init__()
        self.base = base
        self.scale = alpha / rank
        opts = dict(device=base.weight.device, dtype=torch.float32)
        self.lora_a = nn.Parameter(torch.empty(rank, base.in_features, **opts))
        self.lora_b = nn.Parameter(torch.zeros(base.out_features, rank, **opts))
        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))

    def forward(self, x):
        result = self.base(x)
        low = F.linear(x, self.lora_a.to(x.dtype))
        update = F.linear(low, self.lora_b.to(x.dtype))
        return result + self.scale * update


class GigaEncoder(nn.Module):
    def __init__(self, backbone, adapters=True):
        super().__init__()
        self.backbone = backbone.requires_grad_(False)
        self.backbone.config.use_cache = False
        for layer in backbone.layers:
            if layer.self_attn.is_causal:
                raise ValueError("Giga 0826 требует двунаправленного attention.")
            if adapters:
                for target in ("q_proj", "v_proj"):
                    projection = getattr(layer.self_attn, target)
                    setattr(layer.self_attn, target, LoRALinear(projection))
        if adapters:
            backbone.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )

    def forward(self, input_ids, attention_mask):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1)
        mean = (hidden * mask).sum(dim=1, dtype=torch.float32) / mask.sum(dim=1).clamp_min(1)
        return F.normalize(mean, dim=-1)


def load_encoder(model_dir, device, *, adapters=True):
    # This text-only model never uses audio or torchvision. A mismatched optional
    # Kaggle multimedia package must not block the Qwen3 text model import.
    from transformers.utils import import_utils
    import_utils._librosa_available = False
    import_utils._torchvision_available = False
    from transformers import AutoModel, AutoTokenizer

    device = torch.device(device)
    tokenizer = AutoTokenizer.from_pretrained(
        str(model_dir), use_fast=True, local_files_only=True, trust_remote_code=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    dtype = torch.float16 if device.type == "cuda" else torch.float32
    backbone, load_info = AutoModel.from_pretrained(
        str(model_dir), trust_remote_code=True, local_files_only=True,
        use_safetensors=True, dtype=dtype, attn_implementation="sdpa",
        output_loading_info=True
    )
    if any(load_info.get(k) for k in ("missing_keys", "mismatched_keys", "unexpected_keys", "error_msgs")):
        raise RuntimeError(f"Веса Giga не совпадают с архитектурой: {load_info}")
    encoder = GigaEncoder(backbone.to(device), adapters=adapters)
    return tokenizer, encoder.eval()


def tokenize(tokenizer, texts, max_length, device):
    encoded = tokenizer(
        list(texts), padding=True, truncation=True, max_length=max_length,
        pad_to_multiple_of=8 if max_length % 8 == 0 else None,
        return_attention_mask=True, return_tensors="pt"
    )
    return {k: encoded[k].to(device) for k in ("input_ids", "attention_mask")}




@torch.no_grad()
def load_adapter(encoder, state):
    encoder = getattr(encoder, "module", encoder)
    params = {name: p for name, p in encoder.named_parameters() if p.requires_grad}
    if params.keys() != state.keys():
        raise ValueError("Чекпойнт содержит другой набор параметров LoRA.")
    for name, param in params.items():
        if state[name].shape != param.shape:
            raise ValueError(f"Размер параметра LoRA не совпадает: {name}")
        param.copy_(state[name])


## Параметры конкурсного адаптера

Следующая свёрнутая ячейка содержит 144 обученных тензора LoRA шага 306 (2 064 384 числа FP32). SHA-256 проверяется перед чтением весов.


In [ ]:
# Обученные параметры LoRA. Защита от неполной загрузки файла.
import base64, hashlib, zlib
def restore_competition_adapter(path):
    encoded = (
    b"..."
    )
    checkpoint = zlib.decompress(base64.b64decode(encoded, validate=True))
    if hashlib.sha256(checkpoint).hexdigest() != "638288aee64a773f74d7438e16c8c9daa30dfd60770459a46849efb07ec83bc3":
        raise ValueError("Контрольная сумма адаптера не совпала.")
    path.write_bytes(checkpoint)
restore_competition_adapter(WORK / "competition_adapter.pt")


## Обучение и прогноз

На каждом шаге оба GPU накапливают градиенты для попарных сравнений положительных и отрицательных фрагментов одного запроса. Итоговая модель прошла 306 шагов AdamW за два прохода. Косинусное сходство от дообученного энкодера стандартизируется внутри запроса. Его вес в отправке равен 1.


In [ ]:
%%writefile ranker.py
"""Giga 3B query-balanced pairwise ranking. Direct or torchrun (one process/GPU)."""
import argparse
import contextlib
import datetime
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn.functional as F

from data import ndcg, save_submission, within_query
from encoder import load_adapter, load_encoder, query_text, tokenize

SEED = 1949
TEMPERATURE = 0.05
ACCUMULATION = 8
MAX_LENGTH = 512


def distributed():
    return dist.is_available() and dist.is_initialized()


def rank():
    return dist.get_rank() if distributed() else 0


def world():
    return dist.get_world_size() if distributed() else 1


def gather(item):
    if not distributed():
        return [item]
    reports = [None] * world()
    dist.all_gather_object(reports, item)
    return reports


def log(message):
    if rank() == 0:
        print(message, flush=True)


def stopwatch_max(local):
    return max(gather(local))


def autocast(device):
    return torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else contextlib.nullcontext()




def scoring_vectors(texts, tokenizer, model, device, deadline):
    """Encode length-sorted batches, keep vectors in CPU memory, shrink on OOM."""
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)
    vectors = None
    cursor = 0
    batch_cap = 16
    while cursor < len(order):
        if time.time() > deadline:
            raise TimeoutError("Истёк бюджет текущего прохода эмбеддингов.")
        likely_tokens = min(MAX_LENGTH, max(32, len(texts[order[cursor]]) // 2))
        count = max(1, min(batch_cap, 2048 // likely_tokens))
        indices = order[cursor:cursor + count]
        try:
            tokens = tokenize(tokenizer, [texts[i] for i in indices], MAX_LENGTH, device)
            with torch.inference_mode(), autocast(device):
                values = model(**tokens).float().cpu().numpy()
            if not np.isfinite(values).all():
                raise FloatingPointError("Неконечный эмбеддинг Giga.")
        except torch.cuda.OutOfMemoryError:
            if len(indices) == 1:
                raise
            batch_cap = max(1, len(indices) // 2)
            torch.cuda.empty_cache()
            continue
        if vectors is None:
            vectors = np.empty((len(texts), values.shape[1]), dtype=np.float32)
        vectors[indices] = values
        cursor += len(indices)
    return vectors


def score_frame(frame, tokenizer, model, device, deadline):
    """Each rank scores disjoint rows; gather only scalar scores."""
    model.eval()
    group_codes, _ = pd.factorize(frame.query_id, sort=False)
    rows = np.flatnonzero(group_codes % world() == rank())
    local = frame.iloc[rows]
    values = np.empty(len(local), dtype=float)
    try:
        if len(rows):
            qidx, qs = pd.factorize(local["query"], sort=False)
            didx, ds = pd.factorize(local["passage"], sort=False)
            qvec = scoring_vectors([query_text(q) for q in qs], tokenizer, model, device, deadline)
            dvec = scoring_vectors(ds, tokenizer, model, device, deadline)
            for start in range(0, len(local), 256):
                sl = slice(start, start + 256)
                values[sl] = np.einsum("ij,ij->i", qvec[qidx[sl]], dvec[didx[sl]])
        item = (rows, values)
    except TimeoutError:
        item = None
    shards = gather(item)
    if any(x is None for x in shards):
        raise TimeoutError("Время на проход модели истекло.")
    scores = np.empty(len(frame), dtype=float)
    for idx, vals in shards:
        scores[idx] = vals
    return scores


def make_groups(frame, hardness, epoch):
    """All distinct judged negatives are within the same query; positives never compete."""
    rng = np.random.default_rng(SEED + epoch)
    examples = []
    for _, block in frame.groupby("query_id", sort=False):
        pos = block[block.label.eq(1)].drop_duplicates("passage")
        if pos.empty:
            continue
        neg = block[block.label.eq(0) & ~block.passage.isin(pos.passage)].drop_duplicates("passage")
        if neg.empty:
            continue
        shuffled = np.random.default_rng(SEED + int(pos.index[0])).permutation(pos.index)
        positive = np.roll(shuffled, -2 * epoch)[:2]
        article = {str(x) for x in frame.loc[positive, "article_id"] if pd.notna(x) and str(x)}
        eligible = list(neg.index)
        same_article = [i for i in eligible if str(frame.at[i, "article_id"]) in article]
        chosen = []
        if same_article:
            chosen.append(max(same_article, key=lambda i: hardness[i]))
        rest = [i for i in eligible if i not in chosen]
        if rest:
            chosen.append(max(rest, key=lambda i: hardness[i]))
        rest = [i for i in eligible if i not in chosen]
        if rest:
            shuffled_rest = rng.permutation(rest)
            chosen.extend(int(i) for i in shuffled_rest[:3 - len(chosen)])
        examples.append((query_text(block["query"].iloc[0]), list(pos.loc[positive, "passage"]),
                         list(neg.loc[chosen, "passage"])))
    rng.shuffle(examples)
    return examples


def pair_loss(embedding, n_pos):
    similarities = embedding[1:] @ embedding[0]
    margins = similarities[:n_pos, None] - similarities[None, n_pos:]
    return F.softplus(-margins / TEMPERATURE).mean()


def backward_group(group, tokenizer, model, device, scaler, denominator):
    """Gradient caching: replay short blocks, so activations of 6 passages never coexist."""
    query, positive, negative = group
    texts = [query, *positive, *negative]
    blocks = [texts[i:i + 2] for i in range(0, len(texts), 2)]
    tokens = [tokenize(tokenizer, block, MAX_LENGTH, device) for block in blocks]
    # Base attention/LoRA dropout are zero in this pinned model. Save RNG so
    # the reconstruction also matches if a compatible layer uses dropout.
    saved_rng, reps = [], []
    devices = [device.index] if device.type == "cuda" else []
    with torch.no_grad(), autocast(device):
        for batch in tokens:
            saved_rng.append((torch.get_rng_state(), torch.cuda.get_rng_state(device) if devices else None))
            reps.append(model(**batch))
    embedded = torch.cat(reps).detach().requires_grad_(True)
    loss = pair_loss(embedded.float(), len(positive))
    if not torch.isfinite(loss):
        raise FloatingPointError("Неконечный RankNet loss.")
    gradient = torch.autograd.grad(scaler.scale(loss / denominator), embedded)[0]
    offset = 0
    for batch, (cpu_rng, gpu_rng) in zip(tokens, saved_rng):
        with torch.random.fork_rng(devices=devices):
            torch.set_rng_state(cpu_rng)
            if devices:
                torch.cuda.set_rng_state(gpu_rng, device)
            with autocast(device):
                actual = model(**batch)
            next_offset = offset + len(actual)
            actual.backward(gradient[offset:next_offset])
            offset = next_offset
    return float(loss.detach())


def average_gradients(parameters):
    if not distributed():
        return
    gradients = [p.grad for p in parameters if p.grad is not None]
    flat = torch.cat([g.reshape(-1) for g in gradients])
    dist.all_reduce(flat, op=dist.ReduceOp.SUM)
    flat.div_(world())
    offset = 0
    for g in gradients:
        g.copy_(flat[offset:offset + g.numel()].view_as(g))
        offset += g.numel()





def fingerprint_order(frame, scores):
    import hashlib
    ranked = frame[["query_id", "id"]].assign(prediction=scores).sort_values(
        ["query_id", "prediction", "id"], ascending=[True, False, True])
    return hashlib.sha256(("\n".join(ranked.id) + "\n").encode("utf-8")).hexdigest()


def fit_final(work, fit, val, test, model, tok, params, optimizer, scaler, device, deadline):
    lexical = within_query(fit, fit.lex)
    hardness = lexical
    # Restore the original mining budget rule; the submitted report recorded the
    # selected step and weights, but did not record whether neural mining completed.
    t0 = time.time()
    score_frame(val, tok, model, device, deadline)
    val_secs = stopwatch_max(time.time() - t0)
    t0 = time.time()
    score_frame(test, tok, model, device, deadline)
    test_secs = stopwatch_max(time.time() - t0)
    mining_clock_end = float(os.environ["AMML_MINING_CLOCK_END"])
    reserve = 1.40 * (val_secs + test_secs) + 90
    remaining = min(gather(mining_clock_end - time.time())) - reserve
    mining_estimate = max(test_secs * len(fit) / len(test), 1.)
    use_dense = remaining > 0 and mining_estimate * 1.3 < .22 * remaining
    if use_dense:
        try:
            mine_end = min(deadline, mining_clock_end - reserve, time.time() + .22 * remaining)
            mined = score_frame(fit, tok, model, device, mine_end)
        except TimeoutError:
            use_dense = False
        else:
            hardness = .75 * within_query(fit, mined) + .25 * lexical
    log(f"Отбор отрицательных примеров: {'Giga + лексическая модель' if use_dense else 'лексическая модель'}.")

    step = 0
    step_secs = 0.
    for epoch in range(2):
        items = make_groups(fit, hardness, epoch)
        if not items:
            raise ValueError("Нет запросов с обеими метками релевантности.")
        while len(items) % world():
            items.append(items[0])
        local = items[rank()::world()]
        n_steps = math.ceil(len(local) / ACCUMULATION)
        for start in range(0, len(local), ACCUMULATION):
            if min(gather(deadline - time.time())) <= 0:
                raise TimeoutError("Время, выделенное на повторное обучение, истекло.")
            group_batch = local[start:start+ACCUMULATION]
            model.train()
            optimizer.zero_grad(set_to_none=True)
            start_step = time.time()
            warmup = max(1, min(8, n_steps // 10))
            lr_factor = min(1., (step + 1) / warmup) * (.70 if epoch else 1.)
            for group in optimizer.param_groups:
                group["lr"] = 5e-5 * lr_factor
            losses = [backward_group(x, tok, model, device, scaler, len(group_batch)) for x in group_batch]
            average_gradients(params)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.)
            scaler.step(optimizer)
            scaler.update()
            step += 1
            elapsed = stopwatch_max(time.time() - start_step)
            step_secs = elapsed if not step_secs else .75 * step_secs + .25 * elapsed
            if step == 1 or step % 10 == 0:
                log(f"RankNet, проход {epoch+1}/2, шаг {step}, loss {np.mean(losses):.4f}, {elapsed:.1f} с/шаг.")
            if step == 306:
                break
        if step == 306:
            break
    if step != 306 and os.environ["AMML_REFERENCE"] == "1":
        raise ValueError(f"За два прохода получено {step} шагов вместо 306.")
    log(f"Обучение завершено: {step} шагов.")

def run(work, model_dir, device, deadline):
    train = pd.read_pickle(work / "train.pkl")
    val = train.loc[train.fold.eq(0)].reset_index(drop=True)
    fit = train.loc[train.fold.ne(0)].reset_index(drop=True)
    test, sample = (pd.read_pickle(work / name) for name in ("test.pkl", "sample.pkl"))

    torch.manual_seed(SEED)
    tok, model = load_encoder(model_dir, device, adapters=True)
    params = [p for p in model.parameters() if p.requires_grad]
    for p in params:
        dist.broadcast(p.data, src=0)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda", init_scale=128.)
    optimizer = torch.optim.AdamW(params, lr=5e-5, weight_decay=.01)
    phase = os.environ["AMML_PHASE"]
    if phase == "train":
        fit_final(work, fit, val, test, model, tok, params, optimizer, scaler, device, deadline)
    elif phase == "checkpoint":
        state = torch.load(work/"competition_adapter.pt", map_location="cpu", weights_only=True)
        if any(not isinstance(x, torch.Tensor) or x.dtype != torch.float32 or not torch.isfinite(x).all()
               for x in state.values()):
            raise ValueError("Некорректные значения параметров LoRA.")
        # load_adapter checks exact names and shapes against all q/v LoRA parameters in the backbone.
        load_adapter(model, state)
    else:
        raise ValueError(phase)

    # The submitted model used 100% tuned scores and the same rank sharding and batching.
    validation = within_query(val, score_frame(val, tok, model, device, deadline))
    predictions = within_query(test, score_frame(test, tok, model, device, deadline))
    if rank() == 0:
        import hashlib
        score = ndcg(val, validation)
        reference = " (конкурсный запуск 0.950508294935)" if os.environ["AMML_REFERENCE"] == "1" else ""
        print(f"NDCG@10 на {val.query_id.nunique()} отложенных запросах: {score:.12f}{reference}.", flush=True)
        save_submission(test, sample, predictions, work / "submission.csv")
        order_sha = fingerprint_order(test, predictions)
        raw_sha = hashlib.sha256((work/"submission.csv").read_bytes()).hexdigest()
        print(f"SHA-256 порядка документов: {order_sha}. SHA-256 CSV: {raw_sha}.", flush=True)
        (work / "order.sha256").write_text(order_sha)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("work", type=Path)
    parser.add_argument("model")
    parser.add_argument("deadline", type=float)
    args = parser.parse_args()
    local_rank = int(os.environ["LOCAL_RANK"])
    if int(os.environ["WORLD_SIZE"]) != 2:
        raise ValueError("Для воспроизведения требуется две видеокарты.")
    device = torch.device("cuda", local_rank)
    torch.cuda.set_device(device)
    dist.init_process_group("nccl", timeout=datetime.timedelta(minutes=30))
    try:
        run(args.work, args.model, device, args.deadline)
    finally:
        dist.destroy_process_group()


## Запуск на двух GPU

По умолчанию модель обучается на разметке соревнования. Если порядок документов полностью совпадает с отправкой, сохраняется прогноз заново обученной модели. При расхождении или нехватке времени ноутбук рассчитывает предсказания с конкурсными параметрами. Для быстрого запуска с этими параметрами установите `RETRAIN = False` в первой ячейке. Журнал показывает источник прогноза и результат проверки.


In [ ]:
%%writefile prepare.py
"""Select an isolated, compatible Transformers runtime without replacing Kaggle's torch or NumPy."""

import importlib.metadata as metadata
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import zipfile

CORE = {
    "transformers": "4.57.1",
    "huggingface-hub": "0.36.0",
    "tokenizers": "0.22.1",
    "safetensors": "0.6.2",
}
# Runtime dependencies for the four text-model wheels. The notebook's current
# NumPy, torch and CUDA always stay together in the main Kaggle environment.
SUPPORT = (
    "filelock", "fsspec>=2023.5.0", "packaging>=20", "pyyaml>=5.1",
    "regex!=2019.12.17", "requests", "tqdm>=4.27", "typing-extensions>=3.7.4.3",
    "charset-normalizer>=2,<4", "idna>=2.5,<4", "urllib3>=1.21.1,<3", "certifi",
)
PROBE = """
import numpy
import huggingface_hub, safetensors, tokenizers, transformers
from transformers.models.qwen3.configuration_qwen3 import Qwen3Config
assert (transformers.__version__, huggingface_hub.__version__, tokenizers.__version__, safetensors.__version__) == ('4.57.1','0.36.0','0.22.1','0.6.2')
"""


def _environment(extra=None):
    env = dict(os.environ, USE_TF="0", USE_FLAX="0", PIP_DISABLE_PIP_VERSION_CHECK="1")
    if extra is not None:
        paths = [str(extra)] + [p for p in env.get("PYTHONPATH", "").split(os.pathsep) if p and p != str(extra)]
        env["PYTHONPATH"] = os.pathsep.join(paths)
    return env


def _probe(extra=None):
    result = subprocess.run([sys.executable, "-c", PROBE], env=_environment(extra),
                            capture_output=True, text=True, timeout=90)
    return result.returncode == 0, result.stderr[-1800:]


def _activate(target=None):
    os.environ.update({k: v for k, v in _environment(target).items() if k in ("USE_TF", "USE_FLAX", "PYTHONPATH")})
    if target is not None and str(target) not in sys.path:
        sys.path.insert(0, str(target))
    return None if target is None else str(target)


def _support_requirements():
    from packaging.requirements import Requirement
    needed = []
    for item in SUPPORT:
        req = Requirement(item)
        try:
            current = metadata.version(req.name)
        except metadata.PackageNotFoundError:
            current = None
        if current is None or (req.specifier and not req.specifier.contains(current, prereleases=True)):
            needed.append(item)
    return needed


def _local_sources(input_root, tempdir):
    paths, archives, visited = set(), [], set()
    # Kaggle may present datasets or model versions through symlinked directories.
    for folder, dirs, names in os.walk(input_root, followlinks=True):
        real = os.path.realpath(folder)
        if real in visited:
            dirs.clear()
            continue
        visited.add(real)
        if any(n.endswith(".whl") for n in names):
            paths.add(Path(folder))
        archives.extend(Path(folder) / n for n in names if "wheel" in n.lower() and n.lower().endswith(".zip"))
    # Kaggle may mount a ZIP intact or expose its wheels as regular files.
    for archive in sorted(archives):
        with zipfile.ZipFile(archive) as source:
            for entry in source.infolist():
                name = Path(entry.filename).name
                if not name.endswith(".whl"):
                    continue
                dest = tempdir / name
                if not dest.exists():
                    with source.open(entry) as src, dest.open("wb") as out:
                        shutil.copyfileobj(src, out)
                paths.add(tempdir)
    return sorted(paths)


def configure(input_root, cache_dir, allow_download=False):
    """Call once before importing Transformers. New workers inherit the selected PYTHONPATH."""
    ok, _ = _probe()
    if ok:
        print("Transformers 4.57.1 и зависимости уже готовы.", flush=True)
        return _activate()
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    target = cache / f"giga-hf-4.57.1-py{sys.version_info.major}{sys.version_info.minor}"
    if target.is_dir() and _probe(target)[0]:
        return _activate(target)
    specs = [f"{name}=={version}" for name, version in CORE.items()] + _support_requirements()
    errors = []
    with tempfile.TemporaryDirectory(prefix="giga-install-", dir=cache) as temp_name:
        work = Path(temp_name)
        sources = _local_sources(Path(input_root), work)
        modes = []
        if sources:
            flags = ["--no-index"]
            for source in sources:
                flags.extend(["--find-links", str(source)])
            modes.append(flags)
        if allow_download:
            modes.append(["--retries", "1", "--timeout", "15"])
        for index, mode in enumerate(modes):
            stage = work / f"python-{index}"
            install = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                       "--no-cache-dir", "--no-deps", "--only-binary=:all:", "--target", str(stage),
                       *mode, *specs]
            run = subprocess.run(install, env=_environment(), text=True, capture_output=True, timeout=360)
            ok, detail = _probe(stage) if run.returncode == 0 else (False, (run.stderr or run.stdout)[-1800:])
            if ok:
                if target.exists():
                    shutil.rmtree(target)
                os.replace(stage, target)
                print("Подготовлен Transformers 4.57.1, системные torch и NumPy сохранены.", flush=True)
                return _activate(target)
            errors.append(detail)
    detail = "\n".join(errors) if errors else "Набор .whl не найден."
    raise RuntimeError("Подключите amml_transformers_wheels.zip как Kaggle Input или разрешите "
                       "загрузку зависимостей через интернет.\n" + detail)


import json
import os
import sys
from pathlib import Path

import importlib.metadata as metadata
import torch
from data import prepare_data, prepare_inference_inputs
from encoder import prepare_model_dir

if __name__ == "__main__":
    inputs, work = map(Path, sys.argv[1:])
    if torch.cuda.device_count() < 2:
        raise RuntimeError("Запуск рассчитан на две GPU NVIDIA T4.")
    print("GPU:", torch.cuda.get_device_name(0), torch.cuda.get_device_name(1), "CUDA", torch.version.cuda, flush=True)
    packages = configure(inputs, work/"packages", allow_download=True)
    from transformers import AutoConfig
    from transformers.dynamic_module_utils import get_class_from_dynamic_module
    from transformers.utils import import_utils
    for package in ("torch", "transformers", "scikit-learn", "numpy", "pandas", "scipy", "tokenizers"):
        print(package, metadata.version(package), flush=True)
    if os.environ["AMML_RETRAIN"] == "1":
        prepare_data(inputs, work)
    else:
        prepare_inference_inputs(inputs, work)
    # Mounted official weights or the pinned public snapshot. Contest labels stay local.
    cache = Path(os.environ.get("HF_HUB_CACHE", Path(os.environ.get("HF_HOME",
                  Path.home() / ".cache" / "huggingface")) / "hub"))
    model = prepare_model_dir(inputs, cache)
    # The checkpoint loads custom Python. Match the public revision before importing that code.
    import hashlib
    weight_file = Path(model) / "model.safetensors"
    if not weight_file.is_file():
        raise FileNotFoundError("Нужен исходный model.safetensors закреплённой ревизии.")
    with weight_file.open("rb") as file:
        weight_digest = hashlib.file_digest(file, "sha256").hexdigest()
    if weight_digest != "de8519bef7ee360043970b0081088c5294e2a9196ad2d0570a33c4f51bb2e134":
        raise ValueError("Веса базовой модели отличаются от закреплённой ревизии Giga.")
    fingerprints = {
        "config.json": "c97518709921dda9a03309fc90cd53b1fa2d837c8e0afd73e8b32a80dd2ebb82",
        "configuration_qwen3_bidirectional.py": "96e135d1ca3ae77d456b7bbc4973d74277b495f3e664e0c12855f2d642e14b5d",
        "modeling_qwen3_bidirectional.py": "59faa9c3360453700d45697753355f0c70fb19e5768674ffd74d84f1d38775e9",
        "tokenizer.json": "6fb1280bd7fd529f425929b5df823a5a44485cd7fa9679d1ec1acaac4962e8ca",
        "tokenizer_config.json": "843eeba481465c1485a5b5f24bd24d6c12c4e502c16f093c3ab6a0f058c2c5f2",
    }
    for name, expected in fingerprints.items():
        if hashlib.sha256((Path(model)/name).read_bytes()).hexdigest() != expected:
            raise ValueError(f"Файл {name} отличается от закреплённой ревизии Giga.")
    import_utils._librosa_available = False
    import_utils._torchvision_available = False
    auto_map = json.loads((Path(model)/"config.json").read_text())["auto_map"]
    expected_map = {"AutoConfig":"configuration_qwen3_bidirectional.Qwen3BidirectionalConfig",
                    "AutoModel":"modeling_qwen3_bidirectional.Qwen3BidirectionalModel"}
    if any(auto_map.get(k) != v for k, v in expected_map.items()):
        raise ValueError("Неподдерживаемая карта классов архитектуры Giga.")
    config = AutoConfig.from_pretrained(model, trust_remote_code=True, local_files_only=True)
    get_class_from_dynamic_module(config.auto_map["AutoModel"], model, local_files_only=True)
    (work/"model.json").write_text(json.dumps({"model":model, "packages":packages}))


In [ ]:
import hashlib
import json
import platform
import socket
import threading

def run_group(name, commands, environments, deadline):
    """The launcher owns every GPU worker. Reap all workers on error or timeout."""
    if time.monotonic() >= deadline:
        raise TimeoutError(f'Время для этапа «{name}» закончилось до старта.')
    print(f'\n{name}', flush=True)
    processes, readers = [], []

    def forward(stream):
        try:
            for line in stream:
                print(line.rstrip('\n'), flush=True)
        finally:
            stream.close()

    try:
        for command, env in zip(commands, environments, strict=True):
            proc = subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                    text=True, errors='replace', bufsize=1, start_new_session=True)
            processes.append(proc)
            thread = threading.Thread(target=forward, args=(proc.stdout,), daemon=True)
            thread.start()
            readers.append(thread)
        while True:
            codes = [proc.poll() for proc in processes]
            if any(code is not None and code != 0 for code in codes):
                raise RuntimeError(f'Этап «{name}» прерван, коды процессов {codes}. См. вывод выше.')
            if all(code == 0 for code in codes):
                return
            remain = deadline - time.monotonic()
            if remain <= 0:
                raise TimeoutError(f'Достигнут предел времени этапа «{name}».')
            time.sleep(min(1, remain))
    finally:
        # A launcher may already have exited while a child still holds the GPU
        # or its log pipe. Signal the owned process groups independently of the
        # leaders' return codes, then reap the leaders.
        def signal_group(proc, sig):
            try:
                os.killpg(proc.pid, sig)
            except ProcessLookupError:
                return False
            return True

        active = [proc for proc in processes if signal_group(proc, signal.SIGTERM)]
        grace_end = time.monotonic() + 2
        while active and time.monotonic() < grace_end:
            for proc in active:
                proc.poll()
            active = [proc for proc in active if signal_group(proc, 0)]
            if active:
                time.sleep(.05)
        for proc in active:
            signal_group(proc, signal.SIGKILL)
        for proc in processes:
            proc.wait(timeout=2)
        for reader in readers:
            reader.join(timeout=1)

env = dict(os.environ, AMML_RETRAIN=str(int(RETRAIN)), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",
           USE_TF="0", USE_FLAX="0", HF_HUB_DISABLE_TELEMETRY="1",
           OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", HF_MODULES_CACHE=str(WORK / "hf_modules"))
print(platform.platform(), sys.version, flush=True)
try:
    deadline = START + LIMIT
    run_group("Подготовка конкурсных данных и модели", [[sys.executable, str(SRC/"prepare.py"), str(INPUT), str(WORK)]],
              [env], deadline)
    setup = json.loads((WORK/"model.json").read_text())
    if setup["packages"]:
        env["PYTHONPATH"] = os.pathsep.join(filter(None, [setup["packages"], env.get("PYTHONPATH")]))
    # Each rank owns a Unix process group; a failure or timeout terminates both groups.
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        port = str(sock.getsockname()[1])
    rank_envs = [dict(env, LOCAL_RANK=str(i), RANK=str(i), WORLD_SIZE="2",
                      MASTER_ADDR="127.0.0.1", MASTER_PORT=port) for i in range(2)]

    is_official = (WORK/"inputs.sha256").read_text().strip() == "32541df299811d7a7aa788b98633dd2ccfd9c8b82e52d325116646324aba544b"
    def predict(phase, end):
        phase_deadline = time.time() + end - time.monotonic()
        mine_end = time.time() + START + LIMIT - 25*60 - time.monotonic()
        environments = [dict(e, AMML_PHASE=phase, AMML_MINING_CLOCK_END=str(mine_end), AMML_REFERENCE=str(int(is_official))) for e in rank_envs]
        command = [sys.executable, str(SRC/"ranker.py"), str(WORK), setup["model"], str(phase_deadline)]
        (WORK/"order.sha256").unlink(missing_ok=True)
        run_group("Дообучение и прогноз" if phase == "train" else "Прогноз конкурсного адаптера",
                  [command]*2, environments, end)
        return (WORK/"order.sha256").read_text().strip()

    expected_order = "a0166a345bf62f84e0b0c3c3b99f3aae5b8d6a9e8925bdf791e825a4e779f369"
    expected_csv = "def7fc8aa1b0f3970781ee3aee2ff3ed38155ad48006d98dc364e85a7ba3b2ef"
    order = None
    if RETRAIN:
        try:
            order = predict("train", deadline - (60*60 if is_official else 15))
        except (TimeoutError, RuntimeError) as exc:
            if not is_official:
                raise
            print(f"Повторное обучение прервано: {exc}. Использую веса конкурсного запуска.", flush=True)
        if is_official and order and order != expected_order:
            print("Порядок после повторного обучения отличается. Пересчитываю прогноз "
                  "с сохранёнными параметрами конкурсного запуска.", flush=True)
    if order is None or (is_official and order != expected_order):
        order = predict("checkpoint", deadline - 15)
    raw_sha = hashlib.sha256((WORK/"submission.csv").read_bytes()).hexdigest()
    if not is_official:
        print("Подключены другие данные. Сравнение с конкурсной отправкой пропущено.", flush=True)
    elif order == expected_order:
        print("Порядок всех документов внутри запросов совпал с принятой конкурсной отправкой.", flush=True)
    else:
        print("ВНИМАНИЕ: порядок документов отличается даже с конкурсными весами. "
              "Проверьте версии библиотек, данные и вычисления FP16 на двух T4.", flush=True)
    if is_official:
        print(f"Побайтовое совпадение CSV с отправкой: {raw_sha == expected_csv}.", flush=True)
    final_path = OUTPUT / "submission.csv"
    # Atomic publication of the complete, validated table.
    import uuid
    staged = OUTPUT / f".submission-{uuid.uuid4().hex}.tmp"
    try:
        shutil.copyfile(WORK / "submission.csv", staged)
        os.replace(staged, final_path)
    finally:
        staged.unlink(missing_ok=True)
    print(f"Файл для Kaggle — {final_path} ({(time.monotonic()-START)/60:.1f} мин).", flush=True)
finally:
    os.chdir(BASE)
    temp.cleanup()
